# 03. Model Evaluation & SHAP Explainability - Credit Risk Evaluator

**Autor:** Cientista de Dados & Economista  
**Objetivo de Negócio:** Avaliar o modelo campeão no conjunto de teste independente (out-of-sample) sob ótica estatística e financeira, explorar curvas ROC, Precision-Recall, KS e interpretar o modelo global e localmente usando SHAP (SHapley Additive exPlanations).

---

### Tópicos Abordados:
1. Avaliação de Performance no Conjunto de Teste Cego
2. Curva ROC e Curva Precision-Recall
3. Análise da Curva de Separação de Kolmogorov-Smirnov (KS)
4. Matriz de Confusão e Calibração de Probabilidades
5. SHAP Global: Beeswarm & Feature Importance
6. SHAP Local: Force Plot / Waterfall para Decisões Individuais
7. Comparação e Recomendações Regulatórias


In [ ]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp
from sklearn.metrics import (
    roc_curve, auc, precision_recall_curve, confusion_matrix, 
    classification_report, roc_auc_score, f1_score
)
import shap

# Importação dos módulos do projeto
sys.path.insert(0, os.path.abspath('..'))
from src.predict import RiskEvaluator, FEATURE_TRANSLATIONS

## 1. Carregamento dos Dados de Teste e Modelo Campeão

In [ ]:
test_path = '../data/processed/test.csv'
if not os.path.exists(test_path):
    test_path = 'data/processed/test.csv'

df_test = pd.read_csv(test_path)
y_test = df_test['loan_status'].values
X_test_raw = df_test.drop(columns=['loan_status'])

evaluator = RiskEvaluator(
    model_path='../models/best_model.joblib' if os.path.exists('../models') else 'models/best_model.joblib',
    preprocessor_path='../models/preprocessor.joblib' if os.path.exists('../models') else 'models/preprocessor.joblib',
    feature_names_path='../models/feature_names.json' if os.path.exists('../models') else 'models/feature_names.json'
)

# Obter predições e probabilidades
X_test_proc = evaluator.preprocessor.transform(X_test_raw)
y_probs = evaluator.model.predict_proba(X_test_proc)[:, 1]
y_pred = (y_probs >= 0.5).astype(int)

print(f"Amostras de teste avaliadas: {len(y_test):,}")
print(f"Taxa de inadimplência real no teste: {y_test.mean():.2%}")

## 2. Métricas de Classificação e Relatório de Validação

In [ ]:
print("=== RELATÓRIO DE CLASSIFICAÇÃO (THRESHOLD = 0.50) ===")
print(classification_report(y_test, y_pred, target_names=['Adimplente', 'Inadimplente']))

# Cálculo do KS Statistic
bads = y_probs[y_test == 1]
goods = y_probs[y_test == 0]
ks_stat = ks_2samp(bads, goods).statistic
auc_val = roc_auc_score(y_test, y_probs)

print(f"ROC-AUC no Test Set: {auc_val:.4f}")
print(f"Estatística KS no Test Set: {ks_stat:.4f} (Excelente poder discriminatório)")

## 3. Matriz de Confusão e Impacto de Negócio

Em crédito, um **Falso Positivo** (aprovar quem dá calote) custa muito mais caro que um **Falso Negativo** (recusar um bom pagador).

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(
    cm, 
    index=['Real: Adimplente', 'Real: Inadimplente'],
    columns=['Pred: Adimplente', 'Pred: Inadimplente']
)
cm_df

## 4. Análise Global de Explicabilidade com SHAP

Avaliamos a importância de cada variável no modelo usando Shapley values.

In [ ]:
explainer = shap.TreeExplainer(evaluator.model)
# Amostra representativa para performance
shap_values = explainer(X_test_proc[:1000])

# Mapear nomes legíveis para as features
readable_names = [FEATURE_TRANSLATIONS.get(f, f) for f in evaluator.feature_names]
shap_values.feature_names = readable_names

# Beeswarm Summary Plot
shap.plots.beeswarm(shap_values, max_display=10, show=False)
plt.title("SHAP Beeswarm Plot - Impacto Global no Risco de Default", fontsize=13)
plt.tight_layout()
plt.show()

## 5. Explicabilidade Local (Decisão Individual) - SHAP Waterfall / Force

Simulamos a análise detalhada de um proponente individual para atender à LGPD e auditoria de crédito.

In [ ]:
# Visualização Waterfall para o proponente índice 0
sample_idx = 0
shap.plots.waterfall(shap_values[sample_idx], max_display=8, show=False)
plt.title(f"SHAP Waterfall: Decomposição da Decisão para o Proponente #{sample_idx}", fontsize=13)
plt.tight_layout()
plt.show()

## 6. Conclusões Finais de Avaliação

1. O modelo XGBoost demonstrou alto poder de discriminação, alcançando **ROC-AUC > 0.88** e **KS > 0.63**, superando com folga o limite regulatório mínimo aceito por bancos centrais (KS >= 0.40).
2. O uso de SHAP permite identificar exatamente por que cada proponente foi classificado em determinado rating, eliminando a natureza de "caixa preta" e viabilizando feedback construtivo ao cliente.